# Q3 – End-to-End Try-On Inference

Model: CatVTON

GPU: Tesla T4

Preprocessing:
- CatVTON AutoMasker
- Q2 pipeline integration

Outputs:
- output_01.png
- output_02.png
- output_03.png
- output_04.png
- output_05.png

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/XIPL_SDE_Assessment"

Q2_DIR = f"{PROJECT_DIR}/Q2"
Q3_DIR = f"{PROJECT_DIR}/Q3"

import os

os.makedirs(Q3_DIR, exist_ok=True)
os.makedirs(f"{Q3_DIR}/outputs", exist_ok=True)

print(Q2_DIR)
print(Q3_DIR)

/content/drive/MyDrive/XIPL_SDE_Assessment/Q2
/content/drive/MyDrive/XIPL_SDE_Assessment/Q3


In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
Tesla T4


In [ ]:
%cd /content

!rm -rf CatVTON

!git clone https://github.com/Zheng-Chong/CatVTON.git

%cd CatVTON

/content
Cloning into 'CatVTON'...
remote: Enumerating objects: 1358, done.
remote: Counting objects: 100% (309/309), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 1358 (delta 169), reused 128 (delta 128), pack-reused 1049 (from 2)
Receiving objects: 100% (1358/1358), 16.73 MiB | 17.90 MiB/s, done.
Resolving deltas: 100% (462/462), done.
/content/CatVTON


In [ ]:
!pip install -q \
    diffusers==0.30.3 \
    transformers==4.44.2 \
    accelerate==0.30.1 \
    safetensors \
    sentencepiece \
    opencv-python \
    pillow \
    einops \
    huggingface_hub \
    peft==0.11.1 \
    xformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 134.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content

import os

if not os.path.exists("CatVTON"):
    !git clone https://github.com/Zheng-Chong/CatVTON.git

%cd CatVTON

/content
/content/CatVTON


In [ ]:
import torch
import diffusers
import transformers
import peft
import accelerate

print("Torch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU:", torch.cuda.get_device_name(0))

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Torch: 2.11.0+cu128
Diffusers: 0.30.3
Transformers: 4.44.2
PEFT: 0.11.1
Accelerate: 0.30.1
GPU: Tesla T4


In [ ]:
%cd /content/CatVTON

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="zhengchong/CatVTON",
    local_dir="checkpoints/CatVTON",
    local_dir_use_symlinks=False
)

/content/CatVTON


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

mix-48k-1024/attention/model.safetensors:   0%|          | 0.00/198M [00:00<?, ?B/s]

flux-lora/pytorch_lora_weights.safetenso(…):   0%|          | 0.00/37.4M [00:00<?, ?B/s]

vitonhd-16k-512/attention/model.safetens(…):   0%|          | 0.00/198M [00:00<?, ?B/s]

dresscode-16k-512/attention/model.safete(…):   0%|          | 0.00/198M [00:00<?, ?B/s]

'/content/CatVTON/checkpoints/CatVTON'

In [ ]:
import os

print(os.path.exists("checkpoints/CatVTON"))

!find checkpoints/CatVTON -maxdepth 2 -type f | head -20

True
checkpoints/CatVTON/DensePose/Base-DensePose-RCNN-FPN.yaml
checkpoints/CatVTON/DensePose/densepose_rcnn_R_50_FPN_s1x.yaml
checkpoints/CatVTON/DensePose/model_final_162be9.pkl
checkpoints/CatVTON/SCHP/exp-schp-201908261155-lip.pth
checkpoints/CatVTON/SCHP/exp-schp-201908301523-atr.pth
checkpoints/CatVTON/README.md
checkpoints/CatVTON/.gitattributes
checkpoints/CatVTON/flux-lora/pytorch_lora_weights.safetensors
checkpoints/CatVTON/config.json


In [ ]:
%cd /content/CatVTON

import os
import sys

sys.path.append("/content/CatVTON")

print("Current directory:", os.getcwd())

/content/CatVTON
Current directory: /content/CatVTON


In [ ]:
!pip install -q \
fvcore \
iopath \
pyyaml \
omegaconf \
pycocotools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import torch
import fvcore
import iopath

print("Torch :", torch.__version__)
print("fvcore OK")
print("iopath OK")

Torch : 2.11.0+cu128
fvcore OK
iopath OK


In [ ]:
!pip install -q av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 62.5 MB/s eta 0:00:00


In [ ]:
import av
print("PyAV:", av.__version__)

PyAV: 18.0.0


In [ ]:
from model.pipeline import CatVTONPipeline
from model.cloth_masker import AutoMasker
from utils import init_weight_dtype

print("✅ Imports Successful")

✅ Imports Successful


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

weight_dtype = init_weight_dtype("fp16")

pipeline = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt="./checkpoints/CatVTON",
    attn_ckpt_version="mix",
    weight_dtype=weight_dtype,
    device=device,
)

print("✅ CatVTON Pipeline Loaded")

scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

safety_checker/pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/748 [00:00<?, ?B/s]

An error occurred while trying to fetch booksforcharlie/stable-diffusion-inpainting: booksforcharlie/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


unet/diffusion_pytorch_model.bin:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

✅ CatVTON Pipeline Loaded


In [ ]:
from PIL import Image
import os

PROJECT_DIR = "/content/drive/MyDrive/XIPL_SDE_Assessment"

PERSON_DIR = f"{PROJECT_DIR}/Q2/dataset/sample_files/test_pairs/person"
GARMENT_DIR = f"{PROJECT_DIR}/Q2/dataset/sample_files/test_pairs/garment"
MASK_DIR = f"{PROJECT_DIR}/Q2/clothing_masks"
OUTPUT_DIR = f"{PROJECT_DIR}/Q3/outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Person images:", len(os.listdir(PERSON_DIR)))
print("Garment images:", len(os.listdir(GARMENT_DIR)))
print("Masks:", len(os.listdir(MASK_DIR)))

Person images: 5
Garment images: 5
Masks: 5


In [ ]:
person_files = sorted(os.listdir(PERSON_DIR))
garment_files = sorted(os.listdir(GARMENT_DIR))
mask_files = sorted(os.listdir(MASK_DIR))

print(person_files)
print(garment_files)
print(mask_files)

['person_01.png', 'person_02.png', 'person_03.png', 'person_04.png', 'person_05.png']
['garment_01.jpg', 'garment_02.jpg', 'garment_03.jpg', 'garment_04.jpg', 'garment_05.jpg']
['person_01.png', 'person_02.png', 'person_03.png', 'person_04.png', 'person_05.png']


In [ ]:
person = Image.open(os.path.join(PERSON_DIR, person_files[0])).convert("RGB")
cloth = Image.open(os.path.join(GARMENT_DIR, garment_files[0])).convert("RGB")
mask = Image.open(os.path.join(MASK_DIR, mask_files[0])).convert("L")

print(person.size)
print(cloth.size)
print(mask.size)

(768, 1024)
(768, 1024)
(512, 512)


In [ ]:
from PIL import Image

mask = mask.resize(person.size, Image.NEAREST)

print(person.size)
print(cloth.size)
print(mask.size)

(768, 1024)
(768, 1024)
(768, 1024)


In [ ]:
import inspect

print(inspect.signature(pipeline.__call__))

(image: Union[PIL.Image.Image, torch.Tensor], condition_image: Union[PIL.Image.Image, torch.Tensor], mask: Union[PIL.Image.Image, torch.Tensor], num_inference_steps: int = 50, guidance_scale: float = 2.5, height: int = 1024, width: int = 768, generator=None, eta=1.0, **kwargs)


In [ ]:
print(type(result))
print(len(result))

print(type(result[0]))

<class 'list'>
1
<class 'PIL.Image.Image'>


In [ ]:
result[0].save(
    os.path.join(
        OUTPUT_DIR,
        "output_01.png"
    )
)

print("✅ Saved")

✅ Saved


In [ ]:
print(type(result))

if isinstance(result, list):
    print("Length:", len(result))
    print("First element:", type(result[0]))

<class 'list'>
Length: 1
First element: <class 'PIL.Image.Image'>


In [ ]:
result[0].save(f"{OUTPUT_DIR}/output_01.png")

from matplotlib import pyplot as plt

plt.imshow(result[0])
plt.axis("off")
plt.show()

print("✅ output_01.png saved")

✅ output_01.png saved


In [ ]:
print(type(result))

if isinstance(result, list):
    print("Length:", len(result))
    print(type(result[0]))

<class 'list'>
Length: 1
<class 'PIL.Image.Image'>


In [ ]:
result[0].save(f"{OUTPUT_DIR}/output_01.png")

from matplotlib import pyplot as plt

plt.figure(figsize=(6,8))
plt.imshow(result[0])
plt.axis("off")
plt.show()

print("Saved to:", f"{OUTPUT_DIR}/output_01.png")

Saved to: /content/drive/MyDrive/XIPL_SDE_Assessment/Q3/outputs/output_01.png


In [ ]:
print(type(pipeline))
print(hasattr(pipeline, "safety_checker"))

<class 'model.pipeline.CatVTONPipeline'>
True


In [ ]:
pipeline.safety_checker = None

In [ ]:
import inspect

print(inspect.getfile(CatVTONPipeline))

/content/CatVTON/model/pipeline.py


In [ ]:
!sed -n '195,220p' /content/CatVTON/model/pipeline.py


        # Decode the final latents
        latents = latents.split(latents.shape[concat_dim] // 2, dim=concat_dim)[0]
        latents = 1 / self.vae.config.scaling_factor * latents
        image = self.vae.decode(latents.to(self.device, dtype=self.weight_dtype)).sample
        image = (image / 2 + 0.5).clamp(0, 1)
        # we always cast to float32 as this does not cause significant overhead and is compatible with bfloat16
        image = image.cpu().permute(0, 2, 3, 1).float().numpy()
        image = numpy_to_pil(image)
        
        # Safety Check
        if not self.skip_safety_check:
            current_script_directory = os.path.dirname(os.path.realpath(__file__))
            nsfw_image = os.path.join(os.path.dirname(current_script_directory), 'resource', 'img', 'NSFW.jpg')
            nsfw_image = PIL.Image.open(nsfw_image).resize(image[0].size)
            image_np = np.array(image)
            _, has_nsfw_concept = self.run_safety_checker(image=image_np)
            for i,

In [ ]:
pipeline.skip_safety_check = True

In [ ]:
result = pipeline(
    image=person,
    condition_image=cloth,
    mask=mask,
    num_inference_steps=30,
    guidance_scale=2.5,
    height=1024,
    width=768,
)

100%|██████████| 30/30 [01:09<00:00,  2.31s/it]


In [ ]:
result[0].save(f"{OUTPUT_DIR}/output_01.png")

In [ ]:
from PIL import Image
import os

person_files = sorted(os.listdir(PERSON_DIR))
garment_files = sorted(os.listdir(GARMENT_DIR))
mask_files = sorted(os.listdir(MASK_DIR))

for i in range(5):

    print(f"Processing Pair {i+1}/5")

    person = Image.open(
        os.path.join(PERSON_DIR, person_files[i])
    ).convert("RGB")

    cloth = Image.open(
        os.path.join(GARMENT_DIR, garment_files[i])
    ).convert("RGB")

    mask = Image.open(
        os.path.join(MASK_DIR, mask_files[i])
    ).convert("L")

    mask = mask.resize(person.size, Image.NEAREST)

    result = pipeline(
        image=person,
        condition_image=cloth,
        mask=mask,
        num_inference_steps=30,
        guidance_scale=2.5,
        height=1024,
        width=768,
    )

    result[0].save(
        os.path.join(
            OUTPUT_DIR,
            f"output_{i+1:02d}.png"
        )
    )

print("🎉 All 5 outputs generated!")

Processing Pair 1/5


100%|██████████| 30/30 [01:13<00:00,  2.45s/it]


Processing Pair 2/5


100%|██████████| 30/30 [01:17<00:00,  2.57s/it]


Processing Pair 3/5


...

Processing Pair 4/5


100%|██████████| 30/30 [01:17<00:00,  2.58s/it]


Processing Pair 5/5


100%|██████████| 30/30 [01:17<00:00,  2.57s/it]


🎉 All 5 outputs generated!


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(person)
plt.title("Person")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(mask, cmap="gray")
plt.title("Mask")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(cloth)
plt.title("Garment")
plt.axis("off")

plt.show()

In [ ]:
print(type(person))
print(type(mask))
print(type(cloth))

print(person.size)
print(mask.size)
print(cloth.size)

<class 'PIL.Image.Image'>
<class 'PIL.Image.Image'>
<class 'PIL.Image.Image'>
(768, 1024)
(768, 1024)
(768, 1024)


In [ ]:
mask.save("/content/mask_debug.png")

from matplotlib import pyplot as plt

plt.figure(figsize=(5,8))
plt.imshow(mask, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
import numpy as np

mask_np = np.array(mask)

print(mask_np.shape)
print("Unique values:", np.unique(mask_np))

(1024, 768)
Unique values: [  0 255]


In [ ]:
print(MASK_DIR)

/content/drive/MyDrive/XIPL_SDE_Assessment/Q2/clothing_masks


In [ ]:
mask.save("/content/mask.png")
person.save("/content/person.png")
cloth.save("/content/cloth.png")

In [ ]:
mask.save("/content/mask.png")
person.save("/content/person.png")
cloth.save("/content/cloth.png")

print("Saved!")

Saved!


In [ ]:
from model.cloth_masker import AutoMasker
auto_masker = AutoMasker(
    densepose_ckpt="./checkpoints/CatVTON/DensePose",
    schp_ckpt="./checkpoints/CatVTON/SCHP",
    device=device,
)

print("✅ AutoMasker Loaded")

✅ AutoMasker Loaded


In [ ]:
person = Image.open(
    os.path.join(PERSON_DIR, person_files[0])
).convert("RGB")

mask = auto_masker(
    person,
    mask_type="upper"
)

print(type(mask))

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0712 06:53:12.459000 3299 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


<class 'dict'>


In [ ]:
print(type(mask))

if isinstance(mask, dict):
    print(mask.keys())

print(mask)

<class 'dict'>
dict_keys(['mask', 'densepose', 'schp_lip', 'schp_atr'])
{'mask': <PIL.Image.Image image mode=L size=768x1024 at 0x7FB3C4D59D60>, 'densepose': <PIL.Image.Image image mode=L size=768x1024 at 0x7FB3C4DB0D40>, 'schp_lip': <PIL.Image.Image image mode=P size=768x1024 at 0x7FB3C5240860>, 'schp_atr': <PIL.Image.Image image mode=P size=768x1024 at 0x7FB3C5241F70>}


In [ ]:
mask = mask["mask"]

In [ ]:
plt.imshow(mask, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
print(type(mask))

<class 'PIL.Image.Image'>


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,8))
plt.imshow(mask, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
cloth = Image.open(
    os.path.join(GARMENT_DIR, garment_files[0])
).convert("RGB")

result = pipeline(
    image=person,
    condition_image=cloth,
    mask=mask,
    num_inference_steps=30,
    guidance_scale=2.5,
    height=1024,
    width=768,
)

result[0].save(f"{OUTPUT_DIR}/output_01_automask.png")

plt.figure(figsize=(6,8))
plt.imshow(result[0])
plt.axis("off")
plt.show()

100%|██████████| 30/30 [01:01<00:00,  2.07s/it]


In [ ]:
import os

output_path = f"{OUTPUT_DIR}/output_01_automask.png"

print("Exists:", os.path.exists(output_path))
print(output_path)

Exists: True
/content/drive/MyDrive/XIPL_SDE_Assessment/Q3/outputs/output_01_automask.png


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(output_path)

plt.figure(figsize=(6,8))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img1 = Image.open(f"{OUTPUT_DIR}/output_01.png")
img2 = Image.open(f"{OUTPUT_DIR}/output_01_automask.png")

plt.figure(figsize=(12,8))

plt.subplot(1,2,1)
plt.imshow(img1)
plt.title("Using Q2 Mask")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(img2)
plt.title("Using AutoMasker")
plt.axis("off")

plt.show()

In [ ]:
from PIL import Image
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

person_files = sorted(os.listdir(PERSON_DIR))
garment_files = sorted(os.listdir(GARMENT_DIR))

for i in range(5):

    print(f"Processing Pair {i+1}/5")

    # Load images
    person = Image.open(
        os.path.join(PERSON_DIR, person_files[i])
    ).convert("RGB")

    cloth = Image.open(
        os.path.join(GARMENT_DIR, garment_files[i])
    ).convert("RGB")

    # Generate CatVTON mask
    mask_dict = auto_masker(
        person,
        mask_type="upper"
    )

    mask = mask_dict["mask"]

    # Run inference
    result = pipeline(
        image=person,
        condition_image=cloth,
        mask=mask,
        num_inference_steps=30,
        guidance_scale=2.5,
        height=1024,
        width=768,
    )

    # Save output
    save_path = os.path.join(
        OUTPUT_DIR,
        f"output_{i+1:02d}.png"
    )

    result[0].save(save_path)

    print("Saved:", save_path)

print("\n🎉 ALL 5 TRY-ON IMAGES GENERATED!")

Processing Pair 1/5


100%|██████████| 30/30 [01:05<00:00,  2.18s/it]


Saved: /content/drive/MyDrive/XIPL_SDE_Assessment/Q3/outputs/output_01.png
Processing Pair 2/5


100%|██████████| 30/30 [01:16<00:00,  2.55s/it]


Saved: /content/drive/MyDrive/XIPL_SDE_Assessment/Q3/outputs/output_02.png
Processing Pair 3/5


...

Saved: /content/drive/MyDrive/XIPL_SDE_Assessment/Q3/outputs/output_03.png
Processing Pair 4/5


100%|██████████| 30/30 [01:16<00:00,  2.55s/it]


Saved: /content/drive/MyDrive/XIPL_SDE_Assessment/Q3/outputs/output_04.png
Processing Pair 5/5


100%|██████████| 30/30 [01:17<00:00,  2.57s/it]


Saved: /content/drive/MyDrive/XIPL_SDE_Assessment/Q3/outputs/output_05.png

🎉 ALL 5 TRY-ON IMAGES GENERATED!


In [ ]:
import os

print(sorted(os.listdir(OUTPUT_DIR)))

['output_01.png', 'output_01_automask.png', 'output_02.png', 'output_03.png', 'output_04.png', 'output_05.png']


In [ ]:
import os

extra = os.path.join(OUTPUT_DIR, "output_01_automask.png")

if os.path.exists(extra):
    os.remove(extra)

print(sorted(os.listdir(OUTPUT_DIR)))

['output_01.png', 'output_02.png', 'output_03.png', 'output_04.png', 'output_05.png']


In [ ]:
readme = """# Q3 - End-to-End Virtual Try-On Inference

## Objective

This notebook performs virtual try-on using the open-source **CatVTON** model. The pipeline generates realistic try-on images for five person-garment pairs by integrating the preprocessing pipeline developed in Q2.

---

## Model

- Model: CatVTON
- Framework: PyTorch + Diffusers
- GPU: Tesla T4 (Google Colab)
- Precision: FP16

---

## Pipeline

1. Load person and garment images.
2. Generate upper-body mask using CatVTON AutoMasker.
3. Perform virtual try-on using CatVTON.
4. Save the generated output image.

---

## Output

The notebook generates:

Q3/
├── outputs/
│   ├── output_01.png
│   ├── output_02.png
│   ├── output_03.png
│   ├── output_04.png
│   └── output_05.png

---

## Constraints Encountered

### 1. Missing Python Dependencies

Errors encountered:
- fvcore
- iopath
- av

Workaround:
Installed the required packages using pip.

---

### 2. Safety Checker

Issue:
The Stable Diffusion safety checker incorrectly classified generated try-on images as NSFW and replaced them with a placeholder image.

Workaround:
Enabled CatVTON's built-in safety bypass using:

pipeline.skip_safety_check = True

---

### 3. Mask Compatibility

Issue:
Using the clothing masks generated in Q2 produced weak garment transfer where only textures/logos were transferred.

Workaround:
Used CatVTON's AutoMasker to generate the upper-body mask expected by the model during inference.

---

### 4. GPU Memory

GPU:
Tesla T4 (Google Colab)

No out-of-memory errors were encountered after using FP16 inference.

---

## Libraries

- torch
- diffusers
- transformers
- accelerate
- peft
- fvcore
- iopath
- av
- Pillow

---

## Author

Nandana R
B.Tech Computer Science (AI)
"""

with open("/content/drive/MyDrive/XIPL_SDE_Assessment/Q3/README.md", "w") as f:
    f.write(readme)

print("✅ README.md created successfully")

✅ README.md created successfully


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/XIPL_SDE_Assessment"
Q3_DIR = f"{PROJECT_DIR}/Q3"

print(Q3_DIR)

/content/drive/MyDrive/XIPL_SDE_Assessment/Q3
